In [3]:
import numpy as np
import pandas as pd
import sweetviz as sv
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [4]:
adult = pd.read_csv("C:\\Users\\2003n\\Downloads\\adult.csv")
adult.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [5]:
report = sv.analyze(adult)
report.show_html("adult_sweetviz_report.html")

                                             |          | [  0%]   00:00 -> (? left)

Report adult_sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [6]:
adult = adult.replace("?", np.nan)

# strip whitespace in object columns
for col in adult.select_dtypes(include="object").columns:
    adult[col] = adult[col].str.strip()

# binary target
adult["income"] = adult["income"].apply(lambda x: 1 if x == ">50K" else 0)

adult.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0


In [7]:
df = adult.copy()

# 1. capital net
df["capital_net"] = df["capital-gain"] - df["capital-loss"]

# 2. has capital gain flag
df["has_capital_gain"] = (df["capital-gain"] > 0).astype(int)

# 3. overtime flag
df["overtime_flag"] = (df["hours-per-week"] > 40).astype(int)

# 4. age bins
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"]
)

# 5. hours bins
df["hours_group"] = pd.cut(
    df["hours-per-week"],
    bins=[0, 25, 40, 60, 100],
    labels=["part-time", "standard", "heavy", "very-heavy"]
)

# 6. grouped marital status
def group_marital(x):
    if pd.isna(x):
        return "Unknown"
    elif "Married" in x:
        return "Married"
    elif x in ["Never-married"]:
        return "Never-married"
    else:
        return "Previously-married"

df["marital_group"] = df["marital-status"].apply(group_marital)

# 7. grouped education
def group_education(x):
    if pd.isna(x):
        return "Unknown"
    elif x in ["Bachelors", "Masters", "Doctorate", "Prof-school"]:
        return "HigherEd"
    elif x in ["Some-college", "Assoc-acdm", "Assoc-voc"]:
        return "College"
    else:
        return "HS-or-lower"

df["education_group"] = df["education"].apply(group_education)

# 8. interaction-style feature
df["edu_occ_combo"] = df["education_group"].astype(str) + "_" + df["occupation"].fillna("Unknown").astype(str)

df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,...,native-country,income,capital_net,has_capital_gain,overtime_flag,age_group,hours_group,marital_group,education_group,edu_occ_combo
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,...,United-States,0,0,0,0,18-25,standard,Never-married,HS-or-lower,HS-or-lower_Machine-op-inspct
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,...,United-States,0,0,0,1,36-45,heavy,Married,HS-or-lower,HS-or-lower_Farming-fishing
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,...,United-States,1,0,0,0,26-35,standard,Married,College,College_Protective-serv
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,...,United-States,1,7688,1,0,36-45,standard,Married,College,College_Machine-op-inspct
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,...,United-States,0,0,0,0,18-25,standard,Never-married,College,College_Unknown


In [8]:
X = df.drop(columns=["income"])
y = df["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week', 'capital_net', 'has_capital_gain', 'overtime_flag']
Categorical features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country', 'age_group', 'hours_group', 'marital_group', 'education_group', 'edu_occ_combo']


In [9]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [10]:
#Baseline model with logistic regression
log_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

log_pipe.fit(X_train, y_train)

log_pred = log_pipe.predict(X_test)
log_prob = log_pipe.predict_proba(X_test)[:, 1]

print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, log_prob))
print(classification_report(y_test, log_pred))

Logistic Regression Accuracy: 0.8592486436687481
Logistic Regression ROC-AUC: 0.9135618261141942
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.76      0.60      0.67      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.77      0.79      9769
weighted avg       0.85      0.86      0.85      9769



In [11]:
# Hyperparameter tuning Logistic Regression with GridSearchCV
log_param_grid = {
    "model__C": [0.1, 1, 5],
    "model__solver": ["liblinear", "lbfgs"]
}

log_grid = GridSearchCV(
    log_pipe,
    log_param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

log_grid.fit(X_train, y_train)

best_log = log_grid.best_estimator_
best_log_pred = best_log.predict(X_test)
best_log_prob = best_log.predict_proba(X_test)[:, 1]

print("Best Logistic Params:", log_grid.best_params_)
print("Tuned Logistic Accuracy:", accuracy_score(y_test, best_log_pred))
print("Tuned Logistic ROC-AUC:", roc_auc_score(y_test, best_log_prob))
print(classification_report(y_test, best_log_pred))

Best Logistic Params: {'model__C': 1, 'model__solver': 'lbfgs'}
Tuned Logistic Accuracy: 0.8592486436687481
Tuned Logistic ROC-AUC: 0.9135618261141942
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.76      0.60      0.67      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.77      0.79      9769
weighted avg       0.85      0.86      0.85      9769



In [12]:
#Baseline Random Forest model
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42, class_weight="balanced"))
])

rf_pipe.fit(X_train, y_train)

rf_pred = rf_pipe.predict(X_test)
rf_prob = rf_pipe.predict_proba(X_test)[:, 1]

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_prob))
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.8545398710205753
Random Forest ROC-AUC: 0.9001248325196312
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      7431
           1       0.73      0.62      0.67      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.78      0.79      9769
weighted avg       0.85      0.85      0.85      9769



In [13]:
# Hyperparameter tuning Random Forest with GridSearchCV
rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipe,
    rf_param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
best_rf_pred = best_rf.predict(X_test)
best_rf_prob = best_rf.predict_proba(X_test)[:, 1]

print("Best RF Params:", rf_grid.best_params_)
print("Tuned RF Accuracy:", accuracy_score(y_test, best_rf_pred))
print("Tuned RF ROC-AUC:", roc_auc_score(y_test, best_rf_prob))
print(classification_report(y_test, best_rf_pred))

Best RF Params: {'model__max_depth': 20, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Tuned RF Accuracy: 0.8291534445695568
Tuned RF ROC-AUC: 0.9184826321749489
              precision    recall  f1-score   support

           0       0.94      0.83      0.88      7431
           1       0.60      0.83      0.70      2338

    accuracy                           0.83      9769
   macro avg       0.77      0.83      0.79      9769
weighted avg       0.86      0.83      0.84      9769



In [18]:
best_ensemble_prob = best_weight * best_log_prob + (1 - best_weight) * best_rf_prob
best_ensemble_pred = (best_ensemble_prob >= 0.5).astype(int)

print("Ensemble Accuracy:", accuracy_score(y_test, ensemble_pred))
print("Ensemble ROC-AUC:", roc_auc_score(y_test, ensemble_prob))
print(classification_report(y_test, ensemble_pred))

weights = [0.3, 0.4, 0.5, 0.6, 0.7]

best_auc = 0
best_weight = None

for w in weights:
    combined_prob = w * best_log_prob + (1 - w) * best_rf_prob
    auc = roc_auc_score(y_test, combined_prob)
    if auc > best_auc:
        best_auc = auc
        best_weight = w

print("Best logistic weight:", best_weight)
print("Best ensemble ROC-AUC:", best_auc)

Ensemble Accuracy: 0.8578155389497389
Ensemble ROC-AUC: 0.9185150087390822
              precision    recall  f1-score   support

           0       0.91      0.90      0.91      7431
           1       0.70      0.72      0.71      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.81      0.81      9769
weighted avg       0.86      0.86      0.86      9769

Best logistic weight: 0.3
Best ensemble ROC-AUC: 0.9191537911546421


In [19]:
results = pd.DataFrame({
    "Model": [
        "Tuned Logistic Regression",
        "Tuned Random Forest",
        "Equal-Weight Ensemble",
        "Best Weighted Ensemble"
    ],
    "Accuracy": [
        accuracy_score(y_test, best_log_pred),
        accuracy_score(y_test, best_rf_pred),
        accuracy_score(y_test, ensemble_pred),
        accuracy_score(y_test, best_ensemble_pred)
    ],
    "ROC_AUC": [
        roc_auc_score(y_test, best_log_prob),
        roc_auc_score(y_test, best_rf_prob),
        roc_auc_score(y_test, ensemble_prob),
        roc_auc_score(y_test, best_ensemble_prob)
    ]
})

results.sort_values(by="ROC_AUC", ascending=False)



,Model,Accuracy,ROC_AUC
3,Best Weighted Ensemble,0.850445,0.919154
2,Equal-Weight Ensemble,0.857816,0.918515
1,Tuned Random Forest,0.829153,0.918483
0,Tuned Logistic Regression,0.859249,0.913562
